## Cell 1: Setup
**What this demonstrates**: Configure two models — `claude-haiku` for fast, cheap hypothesis generation (vocabulary matching, not reasoning) and `claude-sonnet` for final answer generation. The two-model split is a critical HyDE production decision.
**Expected output**: ✓ Setup complete with both model names, corpus description, and masked key suffixes.

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
%pip install -q langchain langchain-openai langchain-community chromadb python-dotenv

# ── Standard library ─────────────────────────────────────────────────────────
import os
import math
import time
import pathlib
from typing import Any

# ── Third-party ──────────────────────────────────────────────────────────────
from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Load API keys ─────────────────────────────────────────────────────────────
load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'

# ── Constants ─────────────────────────────────────────────────────────────────
# Haiku for hypothesis: fast, cheap — hypothetical only needs vocabulary match, not accuracy
HYDE_MODEL   = 'claude-haiku-4-5-20251001'
# Sonnet for final answer: retrieval is already done; now reasoning quality matters
ANSWER_MODEL = 'claude-sonnet-4-6'
EMBED_MODEL  = 'text-embedding-3-small'
CHROMA_DIR   = './chroma_hyde'
CHUNK_SIZE   = 500

# ── Clients ───────────────────────────────────────────────────────────────────
client:     Anthropic        = Anthropic()
embeddings: OpenAIEmbeddings = OpenAIEmbeddings(model=EMBED_MODEL)

# ── Pure-Python cosine similarity (no numpy dependency) ───────────────────────
def cosine_sim(a: list[float], b: list[float]) -> float:
    """Compute cosine similarity between two embedding vectors.

    Used in Cell 5 to compare how close the query vector vs the hypothetical
    vector is to the top retrieved document — the core HyDE mechanism.
    """
    dot    = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0


print('✓ Setup complete')
print(f'  Hypothesis model: {HYDE_MODEL}')
print(f'  Answer model:     {ANSWER_MODEL}')
print(f'  Embed model:      {EMBED_MODEL}')
print(f'  Anthropic key:    ...{_anthropic_key[-4:]}')
print(f'  OpenAI key:       ...{_openai_key[-4:]}')
print()
print('Corpus: fintech_policy.txt (loan approval criteria, underwriting)')
print('        earnings_report.txt (credit quality metrics, charge-off rates)')
print('Query style: abstract questions — documents style: declarative financial prose')
print('→ Maximum HyDE benefit: wide gap between question and answer vocabulary')

## Cell 2: Data — Loan Policy + Earnings Report
**What this demonstrates**: Build a two-document corpus whose vocabulary gap makes it a strong HyDE test case. Both documents contain credit-related content, but their writing style is radically different from the abstract queries a user would naturally ask.
**Expected output**: Document stats, chunk count, and a concrete vocabulary gap example showing why plain query embedding struggles.

In [ ]:
# ── Locate documents ──────────────────────────────────────────────────────────
_base_candidates = [
    pathlib.Path('../../shared/sample_data'),
    pathlib.Path('shared/sample_data'),
]
BASE_DIR = next((p for p in _base_candidates if p.exists()), None)
assert BASE_DIR is not None, 'Cannot find shared/sample_data.'

DOCS: dict[str, str] = {
    'Meridian Bank Consumer Lending Policy': (BASE_DIR / 'fintech_policy.txt').read_text(encoding='utf-8'),
    'Pinnacle Financial Group Q3 2024 Earnings': (BASE_DIR / 'earnings_report.txt').read_text(encoding='utf-8'),
}

# ── Chunk and index both documents ────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    separators=['\n================================================================================\n', '\n\n', '\n', ' '],
    chunk_size=CHUNK_SIZE,
    chunk_overlap=50,
)

all_docs: list[Document] = []
for doc_title, doc_text in DOCS.items():
    chunks = splitter.split_text(doc_text)
    for i, chunk in enumerate(chunks):
        all_docs.append(Document(
            page_content=chunk,
            metadata={'source': doc_title, 'chunk_idx': i},
        ))

vectorstore = Chroma.from_documents(
    documents=all_docs,
    embedding=embeddings,
    collection_name='hyde_corpus',
    persist_directory=CHROMA_DIR,
)

print('Corpus indexed:')
for title, text in DOCS.items():
    n = sum(1 for d in all_docs if d.metadata['source'] == title)
    print(f'  {title!r}:  {len(text):,} chars  →  {n} chunks')
print(f'  Total: {len(all_docs)} chunks in Chroma')
print()
print('The vocabulary gap this demo exploits:')
print()
print('  User query style (abstract interrogative):')
print('    "What factors influence loan approval decisions?"')
print('    "What are warning signs of credit portfolio stress?"')
print()
print('  Document style (declarative financial prose):')
print('    "All applicants must meet the following minimum requirements..."')
print('    "Net charge-offs in Commercial Banking were $27.8 million (0.18%..."')
print()
print('Plain query embedding: abstract question vector ≠ declarative document vectors')
print('HyDE:                  hypothetical document vector ≈ real document vectors ✓')

## Cell 3: Core — HyDE Pipeline
**What this demonstrates**: The three-function HyDE implementation: `generate_hypothesis` (LLM produces a declarative passage), `embed_hypothesis` (same model as the corpus), and `hyde_retrieve` (searches the corpus with the hypothetical vector, returns real documents). The hypothetical is returned for inspection but never passed to the generation prompt.
**Expected output**: Function definitions confirmed, with a test run showing the hypothesis generation step in isolation.

In [ ]:
# ── Step 1: Generate a hypothetical answer ────────────────────────────────────
def generate_hypothesis(query: str) -> str:
    """Ask the LLM to write a document-style passage that answers the query.

    The prompt instructs the model to write confidently and concretely, in the
    style of a financial policy or earnings document. Hedging is explicitly
    discouraged — a hedged, short response produces a weak embedding that lands
    in the wrong region of the corpus vector space.

    Args:
        query: The user's natural language question.

    Returns:
        A 100–150 word passage in declarative document style.
        This text is NEVER shown to the user — it is a retrieval key only.
    """
    response = client.messages.create(
        model=HYDE_MODEL,
        max_tokens=200,
        system=(
            'You are generating a hypothetical document excerpt for a retrieval system. '
            'Write a 100–150 word passage that directly answers the question, '
            'as if it were an excerpt from a financial policy document or bank earnings report. '
            'Use formal, declarative prose — authoritative statements, specific criteria, '
            'numerical thresholds where plausible. '
            'Do NOT hedge, caveat, or use phrases like "typically" or "may vary". '
            'Write as if you are stating facts from an authoritative source. '
            'The passage will be used only as a search vector — accuracy matters less than style.'
        ),
        messages=[{'role': 'user', 'content': f'Generate a hypothetical document excerpt answering: {query}'}],
    )
    return response.content[0].text.strip()


# ── Step 2: Embed the hypothetical ───────────────────────────────────────────
def embed_hypothesis(hypothesis: str) -> list[float]:
    """Embed the hypothetical answer using the SAME model as the corpus.

    This is a hard requirement: if the corpus was indexed with text-embedding-3-small,
    the hypothesis must also be embedded with text-embedding-3-small. Using a different
    model breaks the shared vector space and makes retrieval meaningless.

    Args:
        hypothesis: The generated document-style passage.

    Returns:
        Embedding vector (1536-dim for text-embedding-3-small).
    """
    return embeddings.embed_query(hypothesis)


# ── Step 3: Retrieve using the hypothetical vector ────────────────────────────
def hyde_retrieve(
    query:      str,
    k:          int = 5,
) -> tuple[str, list[Document]]:
    """Full HyDE pipeline: generate hypothesis → embed → retrieve real documents.

    Args:
        query: The user's original question (unchanged).
        k:     Number of real documents to return.

    Returns:
        (hypothesis_text, real_documents)
        hypothesis_text is returned for inspection and debugging only.
        It must NOT be included in the generation prompt.

    Fintech note:
        On conceptual queries over regulatory or earnings documents, HyDE
        typically outperforms plain retrieval because the hypothesis adopts
        the document's declarative register, placing its vector closer to
        relevant passages in embedding space.
    """
    hypothesis    = generate_hypothesis(query)
    hyp_vector    = embed_hypothesis(hypothesis)
    # Search using the hypothetical vector — NOT the query embedding
    real_docs     = vectorstore.similarity_search_by_vector(hyp_vector, k=k)
    return hypothesis, real_docs


# ── Smoke test: show hypothesis generation in isolation ───────────────────────
test_query = 'What factors influence loan approval decisions?'
print(f'Test query: {test_query!r}')
print()
print('Generating hypothesis (this is the search key — not the answer)...')
t0 = time.perf_counter()
test_hyp = generate_hypothesis(test_query)
hyp_ms   = (time.perf_counter() - t0) * 1000
print()
print('Hypothesis text (will be embedded and used as search vector):')
print('-' * 65)
print(test_hyp)
print('-' * 65)
print(f'Length: {len(test_hyp)} chars  |  ~{len(test_hyp.split())} words  |  Generated in {hyp_ms:.0f} ms')
print()
print('Notice: declarative prose, financial vocabulary, concrete criteria.')
print('This is the vocabulary the corpus was written in — the vector will match.')

## Cell 4: Run — End-to-End HyDE Pipeline
**What this demonstrates**: Full HyDE execution on the loan approval query — hypothesis generation, hypothetical embedding, real document retrieval, and final grounded answer generation from the original query + real documents (never from the hypothesis).
**Expected output**: Generated hypothesis, top-5 real documents retrieved with source attribution, grounded answer citing specific policy sections, and latency breakdown.

In [ ]:
QUERY = 'What factors influence loan approval decisions?'

# ── Step 1–3: HyDE retrieval ──────────────────────────────────────────────────
t0 = time.perf_counter()
hypothetical, hyde_docs = hyde_retrieve(QUERY)
retrieval_ms = (time.perf_counter() - t0) * 1000

# ── Step 4: Generate answer from REAL documents + ORIGINAL query ──────────────
# CRITICAL: hypothetical text is NOT included here.
# The generation prompt sees only the original query and real retrieved documents.
context = '\n\n---\n\n'.join(
    f'[Source {i+1}: {doc.metadata["source"]}]\n{doc.page_content}'
    for i, doc in enumerate(hyde_docs)
)

SYSTEM = (
    'You are a lending policy analyst. '
    'Answer using ONLY the provided document excerpts. '
    'Cite the document name and section for every specific criterion or threshold. '
    'Structure your answer clearly with the key factors listed.'
)

t1 = time.perf_counter()
response = client.messages.create(
    model=ANSWER_MODEL,
    max_tokens=512,
    system=SYSTEM,
    messages=[{'role': 'user', 'content': f'Policy excerpts:\n{context}\n\nQuestion: {QUERY}'}],
)
answer: str = response.content[0].text
gen_ms = (time.perf_counter() - t1) * 1000

# ── Print results ──────────────────────────────────────────────────────────────
print(f'Query: {QUERY!r}')
print()
print(f'Real documents retrieved via hypothetical embedding (top-{len(hyde_docs)}):')
for i, doc in enumerate(hyde_docs, 1):
    src     = doc.metadata['source']
    preview = doc.page_content[:85].replace('\n', ' ')
    print(f'  [{i}] [{src[:38]}]  {preview!r}...')
print()
print('=' * 65)
print('ANSWER  (grounded in real documents — hypothetical never used here)')
print('=' * 65)
print(answer)
print('=' * 65)
print()
print(f'Hypothesis gen:  {retrieval_ms:.0f} ms (incl. embed + retrieve)')
print(f'Answer gen:      {gen_ms:.0f} ms')
print(f'Total:           {retrieval_ms + gen_ms:.0f} ms')

## Cell 5: Inspect — Hypothesis Text and Plain vs HyDE Comparison
**What this demonstrates**: Expose the full hypothesis, compare retrieved document sets between plain and HyDE retrieval, and measure the cosine distance between the query vector vs the hypothetical vector to the top retrieved document — quantifying why HyDE lands closer to the corpus.
**Expected output**: Full hypothesis text, side-by-side document comparison, cosine distance table, and document source attribution showing which corpus each method preferentially retrieves from.

In [ ]:
# ── Show the hypothesis in full ────────────────────────────────────────────────
print('THE HYPOTHETICAL ANSWER (search key — never shown to user):')
print('=' * 65)
print(hypothetical)
print('=' * 65)
print(f'  {len(hypothetical)} chars  |  ~{len(hypothetical.split())} words')
print()
print('This text was embedded and used as the search vector.')
print('It is now discarded. Only the real retrieved documents above remain.')
print()

# ── Plain retrieval for comparison ────────────────────────────────────────────
plain_docs_raw = vectorstore.similarity_search_with_score(QUERY, k=5)
plain_docs = [doc for doc, _ in plain_docs_raw]

# ── Side-by-side comparison ────────────────────────────────────────────────────
print('── RETRIEVAL COMPARISON ─────────────────────────────────────────────────────')
print(f'  {"Rank":<4}  {"Plain (query embedding)":<40}  HyDE (hypothetical embedding)')
print('  ' + '-' * 95)
for rank in range(5):
    plain_col = hyde_col = ''
    if rank < len(plain_docs):
        d = plain_docs[rank]
        src = d.metadata['source'][:18]
        plain_col = f'[{src}]  {d.page_content[:28].replace(chr(10), " ")}...'
    if rank < len(hyde_docs):
        d = hyde_docs[rank]
        src = d.metadata['source'][:18]
        hyde_col = f'[{src}]  {d.page_content[:28].replace(chr(10), " ")}...'
    print(f'  {rank+1:<4}  {plain_col:<40}  {hyde_col}')

# ── Document source attribution ───────────────────────────────────────────────
print()
print('Document source attribution (top-5):')
for label, docs in [('Plain', plain_docs), ('HyDE', hyde_docs)]:
    counts: dict[str, int] = {}
    for d in docs:
        src = d.metadata['source'][:40]
        counts[src] = counts.get(src, 0) + 1
    print(f'  {label}:')
    for src, cnt in sorted(counts.items(), key=lambda x: -x[1]):
        print(f'    {cnt}×  {src!r}')

# ── Cosine distance comparison: query vector vs hypothetical vector ────────────
print()
print('── VECTOR DISTANCE ANALYSIS ─────────────────────────────────────────────────')
print('Measuring how close each search vector is to the top HyDE result...')
print()

# Embed the query and the hypothetical
query_vec = embeddings.embed_query(QUERY)
hyp_vec   = embeddings.embed_query(hypothetical)
# Embed the top retrieved document (its first 500 chars — enough for a representative vector)
top_doc_vec = embeddings.embed_query(hyde_docs[0].page_content[:500])

q_sim   = cosine_sim(query_vec, top_doc_vec)
hyp_sim = cosine_sim(hyp_vec,   top_doc_vec)

print(f'  Top HyDE document (first 80 chars):')
print(f'    {hyde_docs[0].page_content[:80].replace(chr(10), " ")!r}...')
print()
bar_q   = '█' * int(q_sim   * 40)
bar_hyp = '█' * int(hyp_sim * 40)
print(f'  Query vector → doc:       sim={q_sim:.4f}  {bar_q}')
print(f'  Hypothetical vector → doc: sim={hyp_sim:.4f}  {bar_hyp}')
print()
diff = hyp_sim - q_sim
if diff > 0:
    print(f'  Hypothetical is {diff:.4f} closer to the top document than the raw query.')
    print('  → HyDE moved the search vector into the answer region of embedding space. ✓')
else:
    print('  Query vector already close — HyDE benefit is marginal for this query.')
    print('  → Try the risk factor query in Cell 6 for a stronger demonstration.')

## Cell 6: Fintech — Credit Portfolio Risk Factor Search
**What this demonstrates**: HyDE on a conceptual risk query — "What are the warning signs of a deteriorating loan portfolio?" The answer lives in the earnings report's credit quality section (non-accrual loans, charge-off rates, provision increases), written in dense financial reporting prose. The abstract query form is maximally distant from that vocabulary. HyDE bridges the gap.
**Expected output**: Hypothesis in earnings-report register, retrieved credit quality passages from both documents, generated risk analysis, and a demonstration of the vocabulary bridge HyDE creates.

In [ ]:
# ── Conceptual risk query — maximum vocabulary gap ────────────────────────────
# User asks in abstract risk-management language.
# Relevant documents use financial reporting vocabulary:
#   "non-accrual loans", "provision for credit losses", "NCO rate",
#   "allowance for credit losses", "days past due"
# Plain query embedding: abstract ≠ reporting vocabulary → weak retrieval
# HyDE: generates a passage using reporting vocabulary → strong retrieval
RISK_QUERY = 'What are the warning signs of a deteriorating loan portfolio?'

print(f'Query: {RISK_QUERY!r}')
print()

# ── Generate and show the hypothesis ─────────────────────────────────────────
t0 = time.perf_counter()
risk_hypothesis, risk_docs = hyde_retrieve(RISK_QUERY)
hyde_ms = (time.perf_counter() - t0) * 1000

print('Hypothesis (generated in the register of financial reporting prose):')
print('-' * 65)
print(risk_hypothesis)
print('-' * 65)
print()

# ── Show vocabulary bridge ────────────────────────────────────────────────────
# Key terms in the hypothesis that the query did not contain
query_words = set(RISK_QUERY.lower().split())
hyp_words   = set(risk_hypothesis.lower().split())
# Filter for financial reporting terms (longer words, likely domain-specific)
bridged = sorted([w.strip('.,;:()') for w in hyp_words - query_words if len(w) > 6], key=len, reverse=True)[:12]
print(f'Vocabulary bridge — terms in hypothesis NOT in query (domain vocabulary added):')
print(f'  {"  ".join(bridged)}')
print()

# ── Plain retrieval for comparison ────────────────────────────────────────────
plain_risk = vectorstore.similarity_search(RISK_QUERY, k=5)

print('Plain retrieval (query embedding):')
for i, doc in enumerate(plain_risk, 1):
    src = doc.metadata['source'][:40]
    print(f'  [{i}] [{src}]  {doc.page_content[:75].replace(chr(10), " ")!r}...')
print()
print('HyDE retrieval (hypothetical embedding):')
for i, doc in enumerate(risk_docs, 1):
    src = doc.metadata['source'][:40]
    print(f'  [{i}] [{src}]  {doc.page_content[:75].replace(chr(10), " ")!r}...')

# ── Generate risk analysis from real documents ────────────────────────────────
risk_context = '\n\n---\n\n'.join(
    f'[Source {i+1}: {doc.metadata["source"]}]\n{doc.page_content}'
    for i, doc in enumerate(risk_docs)
)

RISK_SYSTEM = (
    'You are a bank credit risk analyst preparing a briefing. '
    'Using ONLY the provided excerpts, identify specific warning signs of '
    'credit portfolio deterioration. '
    'Cite the document and metric name for each indicator. '
    'Include both qualitative triggers and quantitative thresholds where present.'
)

t1 = time.perf_counter()
risk_response = client.messages.create(
    model=ANSWER_MODEL,
    max_tokens=512,
    system=RISK_SYSTEM,
    messages=[{'role': 'user', 'content': f'Excerpts:\n{risk_context}\n\nQuestion: {RISK_QUERY}'}],
)
risk_answer: str = risk_response.content[0].text
gen_ms = (time.perf_counter() - t1) * 1000

print()
print('=' * 65)
print('CREDIT RISK ANALYSIS — Warning Signs')
print('=' * 65)
print(risk_answer)
print('=' * 65)
print()
print('HyDE mechanism summary for this query:')
print('  Query:      abstract risk-management language — no financial metrics')
print('  Hypothesis: earnings-report language — NCO rates, provisions, DPD, ACL ratios')
print('  Result:     real credit quality passages retrieved from the earnings report')
print('              that plain retrieval scored too low to surface')
print()
print(f'HyDE retrieval: {hyde_ms:.0f} ms  |  Answer generation: {gen_ms:.0f} ms  |  Total: {hyde_ms + gen_ms:.0f} ms')